# Silver to Gold — Monthly Recurring Revenue

This notebook converts curated subscription history into business-ready
monthly recurring revenue metrics.

## Outputs

- `gld_customer_monthly_mrr`: one row per customer per month
- `gld_monthly_mrr_summary`: monthly SaaS revenue bridge

## MRR movement definitions

- New MRR: previous month MRR was zero and current month MRR is positive
- Expansion MRR: current MRR increased from the previous month
- Contraction MRR: current MRR decreased but remains positive
- Churned MRR: previous MRR was positive and current MRR is zero

In [ ]:
df = spark.sql("SELECT * FROM lh_ProjPol.dbo.gld_customer_monthly_mrr LIMIT 1000")
display(df)

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers = spark.table("slv_customers")
plans = spark.table("slv_plans")
subscriptions = spark.table("slv_subscriptions")

print(f"Customers:     {customers.count():,}")
print(f"Plans:         {plans.count():,}")
print(f"Subscriptions: {subscriptions.count():,}")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 3, Finished, Available, Finished, False)

Customers:     600
Plans:         4
Subscriptions: 754


In [2]:
#Check the subscription history

display(
    subscriptions
    .orderBy("CustomerID", "StartDate")
    .limit(50)
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c1af44d-f28e-4cb8-bf8b-0520e577d0ec)

In [3]:
#Establish the reporting month range

date_bounds = subscriptions.select(
    F.trunc(F.min("StartDate"), "month").alias("MinMonth"),
    F.trunc(
        F.coalesce(
            F.max("EndDate"),
            F.lit("2026-06-30").cast("date")
        ),
        "month"
    ).alias("MaxMonth")
).first()

min_month = date_bounds["MinMonth"]
max_month = F.to_date(F.lit("2026-06-01"))

print(f"First reporting month: {min_month}")
print("Last reporting month: 2026-06-01")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 5, Finished, Available, Finished, False)

First reporting month: 2023-01-01
Last reporting month: 2026-06-01


In [4]:
#To keep this straightforward, create the month calendar explicitly:
months = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2023-01-01'),
                to_date('2026-06-01'),
                interval 1 month
            )
        ) AS MonthStart
    """)
    .withColumn("MonthEnd", F.last_day("MonthStart"))
)

display(months)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a4051125-9f2d-4815-ab40-362b2601d342)

In [5]:
#Create the customer-month framework

customer_first_subscription = (
    subscriptions
    .groupBy("CustomerID")
    .agg(
        F.trunc(F.min("StartDate"), "month").alias("FirstSubscriptionMonth")
    )
)

customer_months = (
    customer_first_subscription
    .crossJoin(months)
    .filter(F.col("MonthStart") >= F.col("FirstSubscriptionMonth"))
)

print(f"Customer-month combinations: {customer_months.count():,}")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 7, Finished, Available, Finished, False)

Customer-month combinations: 12,855


In [6]:
#Determine the active subscription at each month-end

customer_monthly_mrr = (
    customer_months.alias("cm")
    .join(
        subscriptions.alias("s"),
        (
            (F.col("cm.CustomerID") == F.col("s.CustomerID"))
            & (F.col("s.StartDate") <= F.col("cm.MonthEnd"))
            & (
                F.col("s.EndDate").isNull()
                | (F.col("s.EndDate") >= F.col("cm.MonthEnd"))
            )
        ),
        "left"
    )
    .groupBy(
        F.col("cm.CustomerID"),
        F.col("cm.MonthStart"),
        F.col("cm.MonthEnd")
    )
    .agg(
        F.sum(
            F.coalesce(
                F.col("s.MonthlyRecurringRevenue"),
                F.lit(0)
            )
        ).cast("decimal(18,2)").alias("ClosingMRR")
    )
)

#Using sum() means the logic remains valid even if a customer later has more than one active subscription.

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 8, Finished, Available, Finished, False)

In [7]:
#Add previous-month MRR
customer_month_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy("MonthStart")
)

customer_monthly_mrr = (
    customer_monthly_mrr
    .withColumn(
        "OpeningMRR",
        F.coalesce(
            F.lag("ClosingMRR").over(customer_month_window),
            F.lit(0)
        ).cast("decimal(18,2)")
    )
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 9, Finished, Available, Finished, False)

In [8]:
#Calculate the MRR movement buckets

customer_monthly_mrr = (
    customer_monthly_mrr

    .withColumn(
        "NewMRR",
        F.when(
            (F.col("OpeningMRR") == 0)
            & (F.col("ClosingMRR") > 0),
            F.col("ClosingMRR")
        ).otherwise(F.lit(0))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "ExpansionMRR",
        F.when(
            (F.col("OpeningMRR") > 0)
            & (F.col("ClosingMRR") > F.col("OpeningMRR")),
            F.col("ClosingMRR") - F.col("OpeningMRR")
        ).otherwise(F.lit(0))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "ContractionMRR",
        F.when(
            (F.col("OpeningMRR") > 0)
            & (F.col("ClosingMRR") > 0)
            & (F.col("ClosingMRR") < F.col("OpeningMRR")),
            F.col("OpeningMRR") - F.col("ClosingMRR")
        ).otherwise(F.lit(0))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "ChurnedMRR",
        F.when(
            (F.col("OpeningMRR") > 0)
            & (F.col("ClosingMRR") == 0),
            F.col("OpeningMRR")
        ).otherwise(F.lit(0))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "NetMRRMovement",
        (
            F.col("NewMRR")
            + F.col("ExpansionMRR")
            - F.col("ContractionMRR")
            - F.col("ChurnedMRR")
        ).cast("decimal(18,2)")
    )

    .withColumn(
        "ARR",
        (F.col("ClosingMRR") * 12).cast("decimal(18,2)")
    )
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 10, Finished, Available, Finished, False)

In [9]:
#Add customer attributes

gld_customer_monthly_mrr = (
    customer_monthly_mrr.alias("m")
    .join(
        customers.alias("c"),
        F.col("m.CustomerID") == F.col("c.CustomerID"),
        "left"
    )
    .select(
        F.col("m.CustomerID"),
        F.col("c.CustomerName"),
        F.col("c.Industry"),
        F.col("c.Country"),
        F.col("c.CompanySize"),
        F.col("c.AcquisitionChannel"),
        F.col("m.MonthStart"),
        F.col("m.MonthEnd"),
        F.col("m.OpeningMRR"),
        F.col("m.NewMRR"),
        F.col("m.ExpansionMRR"),
        F.col("m.ContractionMRR"),
        F.col("m.ChurnedMRR"),
        F.col("m.NetMRRMovement"),
        F.col("m.ClosingMRR"),
        F.col("m.ARR")
    )
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 11, Finished, Available, Finished, False)

In [11]:
#Inspect customers with actual movements:

display(
    gld_customer_monthly_mrr
    .filter(
        (F.col("NewMRR") > 0)
        | (F.col("ExpansionMRR") > 0)
        | (F.col("ContractionMRR") > 0)
        | (F.col("ChurnedMRR") > 0)
    )
    .orderBy("CustomerID", "MonthStart")
    .limit(10
    )
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8066f113-9cbb-4f69-aca1-9b2dbd93d8a2)

In [12]:
#Validate the revenue bridge
#Opening MRR
#+ New
#+ Expansion
#- Contraction
#- Churn
#= Closing MRR

bridge_errors = (
    gld_customer_monthly_mrr
    .filter(
        F.abs(
            (
                F.col("OpeningMRR")
                + F.col("NewMRR")
                + F.col("ExpansionMRR")
                - F.col("ContractionMRR")
                - F.col("ChurnedMRR")
            )
            - F.col("ClosingMRR")
        ) > 0.01
    )
)

print(f"Customer-month bridge errors: {bridge_errors.count():,}")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 14, Finished, Available, Finished, False)

Customer-month bridge errors: 0


In [13]:
#Create the monthly executive summary
gld_monthly_mrr_summary = (
    gld_customer_monthly_mrr
    .groupBy("MonthStart", "MonthEnd")
    .agg(
        F.sum("OpeningMRR")
        .cast("decimal(18,2)")
        .alias("OpeningMRR"),

        F.sum("NewMRR")
        .cast("decimal(18,2)")
        .alias("NewMRR"),

        F.sum("ExpansionMRR")
        .cast("decimal(18,2)")
        .alias("ExpansionMRR"),

        F.sum("ContractionMRR")
        .cast("decimal(18,2)")
        .alias("ContractionMRR"),

        F.sum("ChurnedMRR")
        .cast("decimal(18,2)")
        .alias("ChurnedMRR"),

        F.sum("NetMRRMovement")
        .cast("decimal(18,2)")
        .alias("NetMRRMovement"),

        F.sum("ClosingMRR")
        .cast("decimal(18,2)")
        .alias("ClosingMRR"),

        F.sum("ARR")
        .cast("decimal(18,2)")
        .alias("ARR"),

        F.countDistinct(
            F.when(F.col("ClosingMRR") > 0, F.col("CustomerID"))
        ).alias("ActiveCustomers"),

        F.countDistinct(
            F.when(F.col("NewMRR") > 0, F.col("CustomerID"))
        ).alias("NewCustomers"),

        F.countDistinct(
            F.when(F.col("ChurnedMRR") > 0, F.col("CustomerID"))
        ).alias("ChurnedCustomers")
    )
    .withColumn(
        "GrossMRRChurnRate",
        F.when(
            F.col("OpeningMRR") > 0,
            F.col("ChurnedMRR") / F.col("OpeningMRR")
        ).otherwise(F.lit(0))
        .cast("decimal(10,4)")
    )
    .withColumn(
        "NetRevenueRetention",
        F.when(
            F.col("OpeningMRR") > 0,
            (
                F.col("OpeningMRR")
                + F.col("ExpansionMRR")
                - F.col("ContractionMRR")
                - F.col("ChurnedMRR")
            ) / F.col("OpeningMRR")
        ).otherwise(F.lit(None))
        .cast("decimal(10,4)")
    )
    .orderBy("MonthStart")
)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 15, Finished, Available, Finished, False)

In [14]:
display(gld_monthly_mrr_summary)

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9dbd9d4-fbfa-4ad0-9774-c83f56f0f34f)

In [15]:
#Validate the monthly bridge
monthly_bridge_errors = (
    gld_monthly_mrr_summary
    .filter(
        F.abs(
            (
                F.col("OpeningMRR")
                + F.col("NewMRR")
                + F.col("ExpansionMRR")
                - F.col("ContractionMRR")
                - F.col("ChurnedMRR")
            )
            - F.col("ClosingMRR")
        ) > 0.01
    )
)

print(f"Monthly bridge errors: {monthly_bridge_errors.count():,}")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 17, Finished, Available, Finished, False)

Monthly bridge errors: 0


In [16]:
#Write the Gold Delta tables

(
    gld_customer_monthly_mrr.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gld_customer_monthly_mrr")
)

(
    gld_monthly_mrr_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gld_monthly_mrr_summary")
)

print("Gold MRR tables created successfully.")

StatementMeta(, 8273a3db-e5e0-470c-9776-71efdca41fe7, 18, Finished, Available, Finished, False)

Gold MRR tables created successfully.
